# 🎓 Student Performance — Grade Prediction

**Goal:** Predict a student's final math grade (`G3`, scored 0–20) from personal, family, and lifestyle features — *without* using the intermediate grades `G1` and `G2`, which would leak the answer.

**Dataset:** [UCI Student Performance (via Kaggle)](https://www.kaggle.com/datasets/devansodariya/student-performance-data)  
395 students · 33 original columns · Portuguese secondary schools

---

## 📋 Notebook Structure

| # | Section | What it does |
|---|---------|-------------|
| 1 | **Imports** | Load all libraries once |
| 2 | **Load & Clean** | Read CSV, drop leaky/irrelevant columns |
| 3 | **EDA** | Distributions, scatter & box plots |
| 4 | **Correlation Heatmap** | See which features are related to G3 |
| 5 | **Preprocessing** | Encode categoricals, scale, train/test split |
| 6 | **Model Comparison** | Train 8 algorithms, print metrics table |
| 7 | **🎛️ Model Picker** | **Choose which model to test — edit here** |
| 8 | **Overfitting Analysis** | CV gap table + learning curve |
| 9 | **Final Predictions** | Predict G3 on test set, show results |
| 10 | **Feature Importance** | Which features drive the prediction most |

> **Tip:** Every cell imports what it needs locally, so you can re-run any cell independently without restarting the kernel.

## 1. 📦 Install & Import Libraries

We use:
- **pandas / numpy** — data manipulation
- **matplotlib / seaborn** — visualisation
- **scikit-learn** — all ML models, preprocessing, and evaluation metrics

Uncomment the `!pip install` line if you're running this for the first time.

In [ ]:
# !pip install scikit-learn pandas matplotlib seaborn

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     learning_curve, KFold)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

print('✅ All libraries imported successfully')

## 2. 📂 Load & Clean Dataset

The CSV may use a **comma** or **semicolon** separator depending on its source, so we auto-detect it.

### Columns we drop and why

| Column | Reason |
|--------|--------|
| `G1` | 1st period grade — a direct numerical proxy of G3. Including it causes **data leakage**: the model would just learn to copy G1 rather than learning real patterns from behaviour/demographics. |
| `G2` | 2nd period grade — same issue, even stronger correlation with G3 than G1. |
| `school` | Binary identifier (`GP` or `MS`). Encodes which school the student attends, not any behaviour or demographic signal. A model trained with it would not generalise to students from other schools. |

After dropping these we have **29 features** to predict G3.

In [ ]:
import os, pandas as pd

def load_student_data(path='student_data.csv'):
    """Load the student dataset, auto-detecting the separator."""
    if not os.path.exists(path):
        url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/student-mat.csv'
        print(f'File not found locally — downloading from: {url}')
        raw = pd.read_csv(url, sep=None, engine='python')
    else:
        with open(path, 'r') as f:
            first_line = f.readline()
        sep = ';' if ';' in first_line else ','
        print(f'Detected separator: {repr(sep)}')
        raw = pd.read_csv(path, sep=sep)
    if raw.shape[1] == 1:
        raise ValueError('Only 1 column loaded — wrong separator detected.')
    return raw

df_raw = load_student_data('student_data.csv')
print(f'✅ Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

# Drop leaky (G1, G2) and irrelevant (school) columns
COLS_TO_DROP = ['G1', 'G2', 'school']
df = df_raw.drop(columns=COLS_TO_DROP)
print(f'✅ After dropping {COLS_TO_DROP}: {df.shape[1] - 1} features + 1 target (G3)')
df.head()

## 3. 🔍 Exploratory Data Analysis (EDA)

Before building any model, we need to **understand the data**:

- **Are there missing values?** → If yes, we need imputation strategies.
- **What does the target (G3) look like?** → Is it normally distributed? Are there outliers like grade=0 (student didn't sit the exam)?
- **Which features correlate with G3?** → Gives us intuition for feature importance.

The four plots below show:
1. **G3 distribution** — shape of the target variable
2. **Absences vs G3** — do students who miss more classes get lower grades?
3. **Study time vs G3** — does more study time lead to higher grades?
4. **Past failures vs G3** — how strongly do past failures predict future ones?

In [ ]:
import numpy as np

print('Shape:', df.shape)
print('\nMissing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else '  None — dataset is complete ✅')
print('\nG3 (target) statistics:')
print(df['G3'].describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1 — G3 distribution
axes[0,0].hist(df['G3'], bins=21, color='steelblue', edgecolor='white')
axes[0,0].set_title('G3 (Final Grade) Distribution')
axes[0,0].set_xlabel('Grade (0–20)')
axes[0,0].set_ylabel('Number of students')

# 2 — Absences vs G3
axes[0,1].scatter(df['absences'], df['G3'], alpha=0.4, color='steelblue')
axes[0,1].set_title('Absences vs G3')
axes[0,1].set_xlabel('Number of absences')
axes[0,1].set_ylabel('Final Grade (G3)')

# 3 — Study time vs G3 (box plot per category)
st_groups = [df[df['studytime'] == st]['G3'].values for st in sorted(df['studytime'].unique())]
axes[1,0].boxplot(st_groups, labels=sorted(df['studytime'].unique()))
axes[1,0].set_title('Study Time vs G3')
axes[1,0].set_xlabel('Study time (1=<2 h, 2=2–5 h, 3=5–10 h, 4=>10 h)')
axes[1,0].set_ylabel('Final Grade (G3)')

# 4 — Past failures vs G3
fail_groups = [df[df['failures'] == f]['G3'].values for f in sorted(df['failures'].unique())]
axes[1,1].boxplot(fail_groups, labels=sorted(df['failures'].unique()))
axes[1,1].set_title('Past Failures vs G3')
axes[1,1].set_xlabel('Number of past class failures')
axes[1,1].set_ylabel('Final Grade (G3)')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150)
plt.show()
print('📊 EDA plots saved to eda_plots.png')

## 4. 🌡️ Correlation Heatmap

A **correlation heatmap** shows the Pearson correlation coefficient between every pair of numeric features.  
Values range from **−1** (perfect negative) to **+1** (perfect positive). Values near **0** mean no linear relationship.

What to look for:
- **The G3 column/row** — which features correlate most with the target?
- **Feature pairs** — highly correlated features (e.g. `Dalc` ↔ `Walc`) carry redundant information.

Since we removed `G1` and `G2`, the strongest remaining correlations with G3 are expected to be `failures` (negative) and `Medu`/`Fedu` (parental education, positive).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

# Highlight G3 correlations
g3_corr = corr['G3'].drop('G3').sort_values(key=abs, ascending=False)
print('Top features correlated with G3:')
print(g3_corr.to_string())

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Heatmap — numeric features (G1/G2 removed)', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()
print('📊 Heatmap saved to correlation_heatmap.png')

## 5. ⚙️ Preprocessing

Machine learning models require **all-numeric** inputs. The dataset has many categorical (text) columns like `sex`, `address`, `Mjob`, etc.

### Steps performed

1. **Label Encoding** — convert each categorical column to integers.  
   e.g. `sex`: `F` → 0, `M` → 1

2. **Remove G3 = 0 rows** — students with a final grade of 0 didn't sit the exam. These are not true low performers; including them would distort the model.

3. **Train / Test split (80 / 20)** — we train on 80% of the data and hold out 20% as an unseen test set to evaluate real-world performance.

4. **StandardScaler** — rescales every feature to mean=0, std=1.  
   This is critical for distance-based models (KNN, SVR) and regularised linear models (Ridge, Lasso).

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_model = df.copy()

# Step 1 — Encode categorical columns
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include=['object']).columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns: {cat_cols}')
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

# Step 2 — Remove students who did not sit the exam
before = len(df_model)
df_model = df_model[df_model['G3'] > 0].reset_index(drop=True)
print(f'Removed {before - len(df_model)} rows where G3 = 0 (no exam). Remaining: {len(df_model)}')

# Step 3 — Train / test split
X = df_model.drop(columns=['G3'])
y = df_model['G3']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Step 4 — Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'\n✅ Train set: {X_train_sc.shape[0]} samples')
print(f'✅ Test  set: {X_test_sc.shape[0]} samples')
print(f'✅ Features : {X_train_sc.shape[1]}')

## 6. 🏁 Model Comparison — 8 Algorithms

We train and evaluate **8 different regression algorithms** on the same data split so the comparison is fair.

### Algorithms and why each is included

| Model | Type | Why include it |
|-------|------|----------------|
| **Linear Regression** | Linear | Baseline — assumes a straight-line relationship |
| **Ridge** | Linear + L2 regularisation | Reduces overfitting in linear models |
| **Lasso** | Linear + L1 regularisation | Also performs automatic feature selection |
| **Decision Tree** | Non-linear | Captures interactions but tends to overfit |
| **Random Forest** | Ensemble of trees | Reduces Decision Tree overfitting via averaging |
| **Gradient Boosting** | Boosted ensemble | Builds trees sequentially, often very accurate |
| **SVR** | Kernel-based | Good in high-dimensional, small datasets |
| **KNN** | Instance-based | Simple, no training phase, sensitive to scale |

### Metrics
- **RMSE** (Root Mean Squared Error) — penalises large errors more. Lower = better.
- **MAE** (Mean Absolute Error) — average absolute error in grade points. Lower = better.
- **R²** — proportion of variance explained (1.0 = perfect). Higher = better.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

MODELS = {
    'Linear Regression' : LinearRegression(),
    'Ridge'             : Ridge(),
    'Lasso'             : Lasso(),
    'Decision Tree'     : DecisionTreeRegressor(random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR'               : SVR(),
    'KNN'               : KNeighborsRegressor(),
}

results = {}
print(f"{'Model':25s} | {'RMSE':>6} | {'MAE':>6} | {'R²':>6}")
print('-' * 55)
for name, model in MODELS.items():
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    results[name] = {'model': model, 'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f'{name:25s} | {rmse:6.3f} | {mae:6.3f} | {r2:6.3f}')

# Auto-detect best
auto_best_name  = min(results, key=lambda k: results[k]['RMSE'])
auto_best_model = results[auto_best_name]['model']
print(f'\n🏆 Auto-best (lowest RMSE): {auto_best_name}'
      f'  (RMSE={results[auto_best_name]["RMSE"]:.3f}, R²={results[auto_best_name]["R2"]:.3f})')

## 7. 🎛️ Model Picker — Choose Your Model

By default the notebook uses the **auto-best** model (lowest RMSE from Section 6).  
You can override this by changing `SELECTED_MODEL` below to any of the 8 model names.

**Available choices:**
```
'auto'               → use the best model found automatically
'Linear Regression'
'Ridge'
'Lasso'
'Decision Tree'
'Random Forest'
'Gradient Boosting'
'SVR'
'KNN'
```

> **Why would you manually pick a model?**  
> Sometimes you prefer a **simpler, more explainable** model (e.g. Linear Regression) over the most accurate one, especially in academic or reporting contexts.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✏️  CHANGE THIS to test a specific model                       ║
# ║  Options: 'auto', 'Linear Regression', 'Ridge', 'Lasso',       ║
# ║           'Decision Tree', 'Random Forest',                     ║
# ║           'Gradient Boosting', 'SVR', 'KNN'                    ║
# ╚══════════════════════════════════════════════════════════════════╝

SELECTED_MODEL = 'auto'

# ── Resolve the selection ───────────────────────────────────────────
if SELECTED_MODEL == 'auto':
    best_name  = auto_best_name
    best_model = auto_best_model
    print(f'🤖 Auto mode → selected: {best_name}')
elif SELECTED_MODEL in results:
    best_name  = SELECTED_MODEL
    best_model = results[SELECTED_MODEL]['model']
    print(f'✅ Manual selection: {best_name}')
else:
    valid = list(results.keys())
    raise ValueError(
        f'❌ "{SELECTED_MODEL}" is not a valid choice.\n'
        f'   Valid options: {valid}'
    )

r = results[best_name]
print(f'\n📊 {best_name} — Test Set Metrics')
print(f'   RMSE : {r["RMSE"]:.3f}  (avg error in grade points)')
print(f'   MAE  : {r["MAE"]:.3f}  (mean absolute error)')
print(f'   R²   : {r["R2"]:.3f}  (1.0 = perfect fit)')

## 8. 🔬 Overfitting Analysis

**Overfitting** happens when a model memorises the training data instead of learning general patterns. It performs very well on training data but poorly on new (test) data.

We detect it using two complementary methods:

### Method A — Cross-Validation Gap
We run **5-fold cross-validation** on the training set: the training data is split into 5 equal parts, the model is trained on 4 parts and validated on 1, repeated 5 times.  
This gives us a robust estimate of the model's **generalisation error**.  

We then compare it to the **hold-out test RMSE**:
- A small gap (< 0.5) → ✅ model generalises well
- A large gap (> 0.5) → ⚠️ model may be overfitting

### Method B — Learning Curve
We train the selected model on increasing fractions of the training data (10% → 100%) and plot **Train RMSE** vs **Validation RMSE**.

- If both curves **converge** → the model generalises well
- If train RMSE is much lower than validation RMSE throughout → **overfitting**
- If both RMSEs are high → **underfitting** (model is too simple)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':25s} | {'CV RMSE (train)':>16} | {'Test RMSE':>10} | Gap")
print('-' * 75)

for name, info in results.items():
    m = info['model']
    cv_scores = cross_val_score(m, X_train_sc, y_train,
                               cv=kf,
                               scoring='neg_root_mean_squared_error')
    cv_rmse   = -cv_scores.mean()
    test_rmse = info['RMSE']
    gap  = test_rmse - cv_rmse
    flag = '⚠️  overfit?' if gap > 0.5 else '✅'
    marker = ' ◀ SELECTED' if name == best_name else ''
    print(f'{name:25s} | {cv_rmse:16.3f} | {test_rmse:10.3f} | {gap:+.3f}  {flag}{marker}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model,
    X_train_sc, y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='royalblue', label='Train RMSE')
plt.plot(train_sizes, val_rmse,   's-', color='tomato',    label='Validation RMSE')
plt.fill_between(train_sizes,
                 -train_scores.min(axis=1), -train_scores.max(axis=1),
                 alpha=0.1, color='royalblue')
plt.fill_between(train_sizes,
                 -val_scores.min(axis=1), -val_scores.max(axis=1),
                 alpha=0.1, color='tomato')
plt.xlabel('Training set size')
plt.ylabel('RMSE')
plt.title(f'Learning Curve — {best_name}')
plt.legend()
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150)
plt.show()

final_gap = val_rmse[-1] - train_rmse[-1]
print(f'Train RMSE @ full training data : {train_rmse[-1]:.3f}')
print(f'Val   RMSE @ full training data : {val_rmse[-1]:.3f}')
print(f'Gap                             : {final_gap:+.3f}')
if final_gap > 0.5:
    print('⚠️  Noticeable gap — model may be overfitting. Consider reducing complexity or adding regularisation.')
else:
    print('✅ Train/Val gap is acceptable — no significant overfitting detected.')

## 9. 🎯 Final Predictions on Test Set

We now use the **selected model** (from Section 7) to predict G3 for every student in the hold-out test set.

- Predictions are **rounded to the nearest integer** (grades are whole numbers)
- Clipped to the valid range **[0, 20]**
- We show a side-by-side table of Actual vs Predicted vs Error

Two plots are produced:
1. **Actual vs Predicted scatter** — points close to the red dashed line = accurate predictions
2. **Residual plot** — residuals should be randomly scattered around 0. A pattern means the model is systematically wrong for some grade ranges.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Fit on the full training set and predict
best_model.fit(X_train_sc, y_train)
y_pred         = best_model.predict(X_test_sc)
y_pred_rounded = np.clip(np.round(y_pred).astype(int), 0, 20)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f'╔═══════════════════════════════════════╗')
print(f'║  {best_name:^37s} ║')
print(f'╠═══════════════════════════════════════╣')
print(f'║  RMSE : {rmse:6.3f}  (avg error ≈ {rmse:.1f} pts)  ║')
print(f'║  MAE  : {mae:6.3f}                         ║')
print(f'║  R²   : {r2:6.3f}  ({r2*100:.1f}% variance explained) ║')
print(f'╚═══════════════════════════════════════╝')

preds_df = pd.DataFrame({
    'Actual G3'    : y_test.values,
    'Predicted G3' : y_pred_rounded,
    'Error'        : y_pred_rounded - y_test.values
}).reset_index(drop=True)

print('\nFirst 20 predictions:')
print(preds_df.head(20).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Prediction Results — {best_name}', fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='white', s=60)
mn = min(y_test.min(), y_pred.min()) - 0.5
mx = max(y_test.max(), y_pred.max()) + 0.5
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual G3')
axes[0].set_ylabel('Predicted G3')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Plot 2: Residuals
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='tomato', edgecolors='white', s=60)
axes[1].axhline(0, color='black', lw=1.5, linestyle='--')
axes[1].set_xlabel('Predicted G3')
axes[1].set_ylabel('Residual  (Actual − Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('prediction_results.png', dpi=150)
plt.show()
print('📊 Prediction plots saved to prediction_results.png')

## 10. 📊 Feature Importance

**Feature importance** tells us how much each input feature contributed to the model's predictions.

- Available for tree-based models: **Random Forest** and **Gradient Boosting**
- For linear models (Ridge, Lasso, Linear Regression), the **coefficients** serve a similar role
- For SVR and KNN, there is no direct importance score

Higher importance = that feature was used more often (and at higher nodes) to split the data.  
This is useful for understanding which factors (study time, failures, parental education, etc.) drive student performance.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    importances = importances.sort_values(ascending=False)

    plt.figure(figsize=(11, 6))
    colors = ['steelblue' if i < 5 else 'lightsteelblue' for i in range(len(importances))]
    importances.plot(kind='bar', color=colors, edgecolor='white')
    plt.title(f'Feature Importances — {best_name}  (top 5 highlighted)')
    plt.ylabel('Importance score')
    plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150)
    plt.show()
    print('Top 10 most important features:')
    print(importances.head(10).to_string())

elif hasattr(best_model, 'coef_'):
    # Linear models — show coefficients instead
    coefs = pd.Series(np.abs(best_model.coef_), index=X.columns)
    coefs = coefs.sort_values(ascending=False)
    plt.figure(figsize=(11, 6))
    coefs.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title(f'Absolute Coefficients — {best_name}')
    plt.ylabel('|Coefficient|')
    plt.tight_layout()
    plt.savefig('feature_importances.png', dpi=150)
    plt.show()
    print('Top 10 features by coefficient magnitude:')
    print(coefs.head(10).to_string())

else:
    print(f'ℹ️  {best_name} does not expose feature importances or coefficients.')
    print('   Consider switching to Random Forest or Gradient Boosting in the Model Picker (Section 7).')